## Subir modelos

In [ ]:
import os
import re
import shutil
from dotenv import load_dotenv
from huggingface_hub import HfApi

# ============================
# CARGAR TOKEN
# ============================
load_dotenv("./secrets.env")
HF_TOKEN = os.getenv("HF_TOKEN_WRITE")

# ============================
# CONFIGURACIÓN
# ============================
PATH_PAPELERA = "./.Trash-0"

LISTA_CHECKPOINTS = [
    "/notebooks/39000-asturiano-concatenado-instructivo",
    "/notebooks/32000-asturiano-SinConcatenar",
    "/notebooks/33000-aranes-instructivo",
    "/notebooks/37500-asturiano-concatenado",
    "/notebooks/31000-aranes",
    "/notebooks/30000-gallego",
]

api = HfApi()

# ============================
# LIMPIEZA DE PAPELERA
# ============================
def vaciar_papelera_especifica():
    if os.path.exists(PATH_PAPELERA):
        try:
            shutil.rmtree(PATH_PAPELERA)
            os.makedirs(PATH_PAPELERA, exist_ok=True)
        except:
            pass

# ============================
# SUBIR UN QLORA A HF
# ============================
def subir_lora(path_checkpoint):
    nombre_carpeta = path_checkpoint.split("/")[-1]
    nombre_sin_numeros = re.sub(r'^\d+[-_]?', '', nombre_carpeta)
    nombre_limpio = nombre_sin_numeros.lower().replace("_", "-")

    repo_normal = f"MiguelGP-13/{nombre_limpio}"

    print(f"\n==============================")
    print(f" SUBIENDO QLORA: {nombre_limpio}")
    print(f" Repo HF: {repo_normal}")
    print(f"==============================")

    # Crear repo
    api.create_repo(repo_id=repo_normal, token=HF_TOKEN, exist_ok=True)

    # Subir carpeta completa
    api.upload_folder(
        folder_path=path_checkpoint,
        repo_id=repo_normal,
        token=HF_TOKEN
    )

    print(f"✔️ Subido: https://huggingface.co/{repo_normal}")

    vaciar_papelera_especifica()

# ============================
# BUCLE PRINCIPAL
# ============================
if __name__ == "__main__":
    for path in LISTA_CHECKPOINTS:
        if os.path.exists(path):
            subir_lora(path)
        else:
            print(f"Ruta no encontrada: {path}")

    print("\n🚀 ¡Todos los QLoRA han sido subidos a Hugging Face!")



 SUBIENDO QLORA: asturiano-concatenado-instructivo
 Repo HF: MiguelGP-13/asturiano-concatenado-instructivo


adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

optimizer.pt:   0%|          | 0.00/162M [00:00<?, ?B/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.05k [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

✔️ Subido: https://huggingface.co/MiguelGP-13/asturiano-concatenado-instructivo

 SUBIENDO QLORA: asturiano-sinconcatenar
 Repo HF: MiguelGP-13/asturiano-sinconcatenar
✔️ Subido: https://huggingface.co/MiguelGP-13/asturiano-sinconcatenar

 SUBIENDO QLORA: aranes-instructivo
 Repo HF: MiguelGP-13/aranes-instructivo
✔️ Subido: https://huggingface.co/MiguelGP-13/aranes-instructivo

 SUBIENDO QLORA: asturiano-concatenado
 Repo HF: MiguelGP-13/asturiano-concatenado
✔️ Subido: https://huggingface.co/MiguelGP-13/asturiano-concatenado

 SUBIENDO QLORA: aranes
 Repo HF: MiguelGP-13/aranes
✔️ Subido: https://huggingface.co/MiguelGP-13/aranes

 SUBIENDO QLORA: gallego
 Repo HF: MiguelGP-13/gallego
✔️ Subido: https://huggingface.co/MiguelGP-13/gallego

🚀 ¡Todos los QLoRA han sido subidos a Hugging Face!


## Subir datasets de entrenamiento y evaluación

In [2]:
import os
import re
import shutil
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import HfApi

# ============================
# CARGAR TOKEN
# ============================
load_dotenv("./secrets.env")
HF_TOKEN = os.getenv("HF_TOKEN_WRITE")

# ============================
# CONFIGURACIÓN
# ============================
PATH_PAPELERA = "./.Trash-0"
PATH_TEMP = "./_temp_huggingface"

api = HfApi()

def vaciar_papelera_especifica():
    for path in [PATH_PAPELERA, PATH_TEMP]:
        if os.path.exists(path):
            try:
                shutil.rmtree(path)
            except:
                pass
    os.makedirs(PATH_TEMP, exist_ok=True)

def preparar_y_subir(repo_name, archivos_a_procesar, es_txt=False):
    """
    Recibe un diccionario de {nombre_archivo_final: ruta_origen}.
    Lee CSVs directamente (si no es txt), convierte a Parquet y los sube a HF.
    """
    repo_dataset = f"MiguelGP-13/{repo_name.lower().replace('_', '-')}"
    carpeta_envio = os.path.join(PATH_TEMP, repo_name)
    os.makedirs(carpeta_envio, exist_ok=True)
    
    print(f"\n==================================================")
    print(f" PREPARANDO DATASET: {repo_dataset}")
    print(f"==================================================")

    for nombre_final, ruta_origen in archivos_a_procesar.items():
        ruta_destino = os.path.join(carpeta_envio, nombre_final)
        
        if es_txt:
            shutil.copy(ruta_origen, ruta_destino)
            print(f"   -> Copiado TXT: {nombre_final}")
        else:
            try:
                df = pd.read_csv(ruta_origen)
                df.to_parquet(ruta_destino, index=False)
                print(f"   -> Convertido CSV a Parquet: {nombre_final}")
            except Exception as e:
                print(f"   ❌ Error procesando el CSV {ruta_origen}: {e}")
                return

    api.create_repo(repo_id=repo_dataset, token=HF_TOKEN, repo_type="dataset", exist_ok=True)
    api.upload_folder(
        folder_path=carpeta_envio,
        repo_id=repo_dataset,
        token=HF_TOKEN,
        repo_type="dataset"
    )
    print(f"✔️ Dataset disponible en: https://huggingface.co/datasets/{repo_dataset}")

# ============================
# PROCESAMIENTO POR SECCIÓN
# ============================

def procesar_eval_datasets():
    print("\n🔍 Escaneando EvalDatasets...")
    ruta_eval = "/notebooks/EvalDatasets"
    if not os.path.exists(ruta_eval): return

    for subcarpeta in os.listdir(ruta_eval):
        if subcarpeta.lower() == "raw": 
            continue
            
        ruta_sub = os.path.join(ruta_eval, subcarpeta)
        if os.path.isdir(ruta_sub):
            for archivo in os.listdir(ruta_sub):
                ruta_archivo = os.path.join(ruta_sub, archivo)
                if os.path.isfile(ruta_archivo):
                    nombre_base, _ = os.path.splitext(archivo)
                    repo_name = f"eval-{subcarpeta.lower()}-{nombre_base.lower()}"
                    archivos_subida = {f"{nombre_base.lower()}.parquet": ruta_archivo}
                    preparar_y_subir(repo_name, archivos_subida)

def procesar_train_datasets():
    print("\n🔍 Escaneando TrainDatasets (Agrupando por Lengua)...")
    ruta_train = "/notebooks/TrainDatasets"
    if not os.path.exists(ruta_train): return

    carpetas_objetivo = ["Instructivo", "Instructivo_QA"]
    agrupacion_por_lengua = {}
    
    for objetivo in carpetas_objetivo:
        ruta_obj = os.path.join(ruta_train, objetivo)
        if not os.path.exists(ruta_obj): continue
        
        for archivo in os.listdir(ruta_obj):
            ruta_archivo = os.path.join(ruta_obj, archivo)
            if os.path.isfile(ruta_archivo):
                lengua, _ = os.path.splitext(archivo)
                lengua = lengua.lower()
                
                if lengua not in agrupacion_por_lengua:
                    agrupacion_por_lengua[lengua] = {}
                
                nombre_archivo_final = f"{objetivo.lower()}.parquet"
                agrupacion_por_lengua[lengua][nombre_archivo_final] = ruta_archivo

    for lengua, archivos_subida in agrupacion_por_lengua.items():
        repo_name = f"instructivo-{lengua}"
        preparar_y_subir(repo_name, archivos_subida)

def procesar_lexicons():
    print("\n🔍 Escaneando Lexicons...")
    ruta_lexicons = "/notebooks/lexicons"
    if not os.path.exists(ruta_lexicons): return

    archivos_lexicons = {}
    for archivo in os.listdir(ruta_lexicons):
        ruta_archivo = os.path.join(ruta_lexicons, archivo)
        if os.path.isfile(ruta_archivo) and archivo.endswith('.txt'):
            archivos_lexicons[archivo.lower()] = ruta_archivo

    if archivos_lexicons:
        preparar_y_subir("lexicons-all", archivos_lexicons, es_txt=True)

# ============================
# BUCLE PRINCIPAL
# ============================
if __name__ == "__main__":
    vaciar_papelera_especifica()
    
    procesar_eval_datasets()
    procesar_train_datasets()
    procesar_lexicons()
    
    vaciar_papelera_especifica()
    print("\n🚀 ¡Estructura definitiva subida con éxito!")


🔍 Escaneando EvalDatasets...

 PREPARANDO DATASET: MiguelGP-13/eval-huecos-aranes
   -> Convertido CSV a Parquet: aranes.parquet
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/eval-huecos-aranes

 PREPARANDO DATASET: MiguelGP-13/eval-huecos-gallego
   -> Convertido CSV a Parquet: gallego.parquet
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/eval-huecos-gallego

 PREPARANDO DATASET: MiguelGP-13/eval-huecos-asturiano
   -> Convertido CSV a Parquet: asturiano.parquet
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/eval-huecos-asturiano

 PREPARANDO DATASET: MiguelGP-13/eval-anotado-aranes
   -> Convertido CSV a Parquet: aranes.parquet
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/eval-anotado-aranes

 PREPARANDO DATASET: MiguelGP-13/eval-anotado-gallego
   -> Convertido CSV a Parquet: gallego.parquet
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/eval-anotado-gallego

 PREP

instructivo_qa.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/instructivo-aranes

 PREPARANDO DATASET: MiguelGP-13/instructivo-gallego
   -> Convertido CSV a Parquet: instructivo.parquet
   -> Convertido CSV a Parquet: instructivo_qa.parquet


instructivo_qa.parquet:   0%|          | 0.00/204k [00:00<?, ?B/s]

✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/instructivo-gallego

 PREPARANDO DATASET: MiguelGP-13/instructivo-asturiano
   -> Convertido CSV a Parquet: instructivo.parquet
   -> Convertido CSV a Parquet: instructivo_qa.parquet


instructivo_qa.parquet:   0%|          | 0.00/176k [00:00<?, ?B/s]

✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/instructivo-asturiano

🔍 Escaneando Lexicons...

 PREPARANDO DATASET: MiguelGP-13/lexicons-all
   -> Copiado TXT: fr.txt
   -> Copiado TXT: gl.txt
   -> Copiado TXT: es.txt
   -> Copiado TXT: ast.txt
   -> Copiado TXT: aran.txt
✔️ Dataset disponible en: https://huggingface.co/datasets/MiguelGP-13/lexicons-all

🚀 ¡Estructura definitiva subida con éxito!
